In [48]:
import pandas as pd  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import os

# Path to the dataset
dataset_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Dataset_Creator\poses_dataset.csv'
df = pd.read_csv(dataset_path)

def calculate_angle(a, b, c):
    """Calcula o ângulo em graus entre três pontos (a, b, c) onde b é o vértice."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180.0 else angle

In [49]:
# 1. Feature Engineering & Data Preparation
def extract_advanced_features(row):
    # Only use important keypoints: Hips(11,12), Knees(13,14), Ankles(15,16)
    important_indices = [11, 12, 13, 14, 15, 16]
    
    # --- GEOMETRIA ESTÁTICA ---
    # Hip center for normalization (Standard practice in HAR research, e.g., Haider et al., 2023)
    hip_center = np.array([
        (row['kp11_x'] + row['kp12_x']) / 2,
        (row['kp11_y'] + row['kp12_y']) / 2
    ])
    
    relative_coords = []
    for idx in important_indices:
        px = row[f'kp{idx}_x'] - hip_center[0]
        py = row[f'kp{idx}_y'] - hip_center[1]
        relative_coords.extend([px, py])
        
    angle_knee_l = calculate_angle([row['kp11_x'], row['kp11_y']], [row['kp13_x'], row['kp13_y']], [row['kp15_x'], row['kp15_y']])
    angle_knee_r = calculate_angle([row['kp12_x'], row['kp12_y']], [row['kp14_x'], row['kp14_y']], [row['kp16_x'], row['kp16_y']])
    angle_hip_l = calculate_angle([row['kp5_x'], row['kp5_y']], [row['kp11_x'], row['kp11_y']], [row['kp13_x'], row['kp13_y']])
    angle_hip_r = calculate_angle([row['kp6_x'], row['kp6_y']], [row['kp12_x'], row['kp12_y']], [row['kp14_x'], row['kp14_y']])
    
    dist_foot_l = np.linalg.norm(np.array([row['kp15_x'], row['kp15_y']]) - hip_center)
    dist_foot_r = np.linalg.norm(np.array([row['kp16_x'], row['kp16_y']]) - hip_center)
    
    return relative_coords + [angle_knee_l, angle_knee_r, angle_hip_l, angle_hip_r, dist_foot_l, dist_foot_r]

print("Extracting advanced features (Based on Haider et al., 2023 methodology)...")
features_list = df.apply(extract_advanced_features, axis=1).tolist()
X_advanced = np.array(features_list)
y_raw = df['label'].values

# --- SLIDING WINDOW IMPLEMENTATION ---
# Based on spatial-temporal feature aggregation used in Human Activity Recognition (HAR)
WINDOW_SIZE = 30
X_windows, y_windows = [], []

for video in df['video_name'].unique():
    mask = df['video_name'] == video
    video_data = X_advanced[mask]
    video_labels = y_raw[mask]
    
    for i in range(0, len(video_data) - WINDOW_SIZE):
        window = video_data[i : i + WINDOW_SIZE]
        X_windows.append(window.flatten())
        y_windows.append(1 if np.any(video_labels[i : i + WINDOW_SIZE] == 1) else 0)

X = np.array(X_windows)
y = np.array(y_windows)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(f"Final Dataset Shape: {X.shape}")

Extracting advanced features (Based on Haider et al., 2023 methodology)...
Final Dataset Shape: (1505, 540)


In [50]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [51]:
def build_kick_model(input_shape):
    model = models.Sequential([
        # Input Layer -> Hidden Layer 1
        layers.Dense(512, activation='relu', input_shape=(input_shape,), kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.3),

        # Hidden Layer 2
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.3),

        # Hidden Layer 3
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.3),

        # Hidden Layer 4
        layers.Dense(64, activation='relu'),

        # Output Layer (Sigmoid for binary 0/1 classification)
        layers.Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [52]:
early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,          # Wait 10 epochs before giving up
      restore_best_weights=True # Give me the model from the best epoch, not the last one
  )

In [53]:
nn_model = build_kick_model(X_train.shape[1])

nn_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stop])

print(nn_model.evaluate(X_test, y_test))

Epoch 1/100


c:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\.venv\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8702 - loss: 1.3410 - val_accuracy: 0.9502 - val_loss: 1.1332
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9294 - loss: 1.1452 - val_accuracy: 0.9751 - val_loss: 1.0309
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9574 - loss: 1.0211 - val_accuracy: 0.9668 - val_loss: 1.0113
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9688 - loss: 0.9616 - val_accuracy: 0.9834 - val_loss: 0.8936
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9803 - loss: 0.8843 - val_accuracy: 0.9751 - val_loss: 0.8758
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9720 - loss: 0.8562 - val_accuracy: 0.9834 - val_loss: 0.8081
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9875 - loss: 0.7733 - val_accuracy: 0.9793 - val_loss: 0.7530
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9875 - loss: 0.7158 - val_accuracy: 0.9793 - val_loss: 0.

In [54]:
def evaluate_model(name, model, X_test, y_test):
      # For NN, we need to convert probabilities to 0 or 1
      if hasattr(model, 'predict_proba') or 'RandomForest' in str(type(model)):
          preds = model.predict(X_test)
      else: # TensorFlow model
          probs = model.predict(X_test)
          preds = (probs > 0.5).astype(int).flatten()

      print(f"\n--- {name} Report ---")
      print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
      print(classification_report(y_test, preds))

In [55]:
evaluate_model("Neural Network", nn_model, X_test, y_test)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 

--- Neural Network Report ---
Accuracy: 0.9900
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       205
           1       0.98      0.99      0.98        96

    accuracy                           0.99       301
   macro avg       0.99      0.99      0.99       301
weighted avg       0.99      0.99      0.99       301



In [56]:
# Save model and scaler to the correct folder
model_save_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\kick_detection_model.h5'
scaler_save_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\scaler.pkl'

nn_model.save(model_save_path)
joblib.dump(scaler, scaler_save_path)
print(f"Model saved to {model_save_path}")
print(f"Scaler saved to {scaler_save_path}")

Model saved to C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\kick_detection_model.h5
Scaler saved to C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\scaler.pkl
